In [1]:
import os
import shutil
import numpy as np
from config import Config
from env import MySumoEnv

_NEEDED_WORKER_FILES = ("detectors.add.xml", "network.net.xml", "run.sumocfg")

def prepare_worker_dir(worker_idx=0):
    if hasattr(Config, "ensure_dirs"):
        Config.ensure_dirs()
    else:
        os.makedirs(Config.WORKERS_ROOT, exist_ok=True)
    src = Config.BASE_WORK_DIR
    dst = Config.worker_dir(worker_idx)
    os.makedirs(dst, exist_ok=True)
    for name in _NEEDED_WORKER_FILES:
        src_path = os.path.join(src, name)
        dst_path = os.path.join(dst, name)
        if not os.path.exists(src_path):
            raise FileNotFoundError(f"No utils file: {src_path}")
        shutil.copy2(src_path, dst_path)
    os.makedirs(os.path.join(dst, "dump"), exist_ok=True)
    return dst

def build_env(worker_idx=0, total_step=None, seed=42):
    rl_dir = prepare_worker_dir(worker_idx)
    answer_path = Config.answer_path_for(worker_idx)
    sumocfg_path = os.path.join(rl_dir, "run.sumocfg")

    if not os.path.exists(rl_dir):
        raise FileNotFoundError(f"No directory: {rl_dir}")
    if not os.path.exists(sumocfg_path):
        raise FileNotFoundError(f"No run.sumocfg: {sumocfg_path}")
    if not os.path.exists(answer_path):
        raise FileNotFoundError(f"No answer.csv: {answer_path}")

    if total_step is None:
        total_step = Config.TOTAL_STEP

    env = MySumoEnv(
        rl_dir=rl_dir,
        sumo_binary=Config.SUMO_BINARY,
        origin_list=Config.ORIGIN_LIST,
        destination_list=Config.DESTINATION_LIST,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        num_OD=Config.NUM_OD,
        state_dim=1,
        answer_dir=answer_path,
        total_step=int(total_step),
    )

    obs, info = env.reset(seed=seed)
    return env, obs, info

In [2]:
import time
from trial_timing import reset_trial_times, append_trial_time

import os
import json
import numpy as np
from config import Config

def evenly_binary_sequence(count: int, n: int) -> np.ndarray:
\
\
       
    count = int(np.clip(count, 0, n))
    seq = np.zeros(n, dtype=np.int32)
    if count == 0:
        return seq
    if count == n:
        seq[:] = 1
        return seq

    acc = 0
    for i in range(n):
        acc += count
        if acc >= n:
            seq[i] = 1
            acc -= n
    return seq

def counts_to_binary_action_plan(
    counts_block: np.ndarray,
    num_od: int,
    total_step: int,
    block_steps: int
) -> np.ndarray:
\
\
\
       
    n_blocks = int(np.ceil(total_step / block_steps))
    action_plan = np.zeros((total_step, num_od), dtype=np.int32)

    for b in range(n_blocks):
        s = b * block_steps
        e = min((b + 1) * block_steps, total_step)
        n_local = e - s
        for od in range(num_od):
            c = int(np.rint(counts_block[b, od]))
            c = int(np.clip(c, 0, n_local))
            action_plan[s:e, od] = evenly_binary_sequence(c, n_local)

    return action_plan

def run_episode_with_plan(build_env_fn, worker_idx, action_plan, seed):
    env, obs, info = build_env_fn(
        worker_idx=worker_idx,
        total_step=action_plan.shape[0],
        seed=seed
    )
    total_reward = 0.0
    last_info = {}

    try:
        for t in range(action_plan.shape[0]):
            obs, reward, terminated, truncated, info = env.step(action_plan[t])
            total_reward += float(reward)
            last_info = info
            if terminated or truncated:
                break
    finally:
        env.close()

    trajectory = last_info.get("trajectory", [])
    return float(total_reward), trajectory

def spsa_optimize_counts_300s(
    build_env_fn,
    worker_idx,
    num_od,
    total_step,
    input_interval,
    detector_interval,
    init_points=0,
    n_iter=200,
    seed=42,
    verbose=2,
    alpha=0.602,
    gamma=0.101,
    a=3.0,
    c=2.0,
    A=None,
):
    block_steps = detector_interval // input_interval
    n_blocks = int(np.ceil(total_step / block_steps))

    var_meta = []
    lb = []
    ub = []
    for b in range(n_blocks):
        s = b * block_steps
        e = min((b + 1) * block_steps, total_step)
        n_local = e - s
        for od in range(num_od):
            var_meta.append((b, od, n_local))
            lb.append(0.0)
            ub.append(float(n_local))

    lb = np.array(lb, dtype=np.float32)
    ub = np.array(ub, dtype=np.float32)
    dim = lb.size

    rng = np.random.default_rng(seed)
    eval_count = {"n": 0}

    def x_to_counts(x_vec: np.ndarray) -> np.ndarray:
        x_vec = np.asarray(x_vec, dtype=np.float32)
        counts = np.zeros((n_blocks, num_od), dtype=np.float32)
        for i, (b, od, n_local) in enumerate(var_meta):
            counts[b, od] = float(np.clip(x_vec[i], 0.0, float(n_local)))
        return counts

    def evaluate_x(x_vec: np.ndarray):
        x_vec = np.clip(np.asarray(x_vec, dtype=np.float32), lb, ub)
        counts_block = x_to_counts(x_vec)
        action_plan = counts_to_binary_action_plan(
            counts_block=counts_block,
            num_od=num_od,
            total_step=total_step,
            block_steps=block_steps
        )
        score, _ = run_episode_with_plan(
            build_env_fn=build_env_fn,
            worker_idx=worker_idx,
            action_plan=action_plan,
            seed=seed + eval_count["n"],
        )
        eval_count["n"] += 1
        return float(score)

    x = 0.5 * (lb + ub)

    best_score = -np.inf
    best_x = x.copy()

    history = []

    for i in range(int(init_points)):
        xr = rng.uniform(lb, ub).astype(np.float32)
        yr = evaluate_x(xr)
        history.append({"eval": int(eval_count["n"]), "type": "init", "score": float(yr)})
        if yr > best_score:
            best_score = yr
            best_x = xr.copy()

    if int(init_points) > 0:
        x = best_x.copy()

    eval_budget = int(max(0, n_iter))
    spsa_steps = eval_budget // 2
    leftover_eval = eval_budget % 2

    if A is None:
        A = max(10.0, 0.1 * max(1, spsa_steps))

    for k in range(1, spsa_steps + 1):
        ak = a / ((k + A) ** alpha)
        ck = c / (k ** gamma)

        delta = rng.choice(np.array([-1.0, 1.0], dtype=np.float32), size=dim)
        x_plus  = np.clip(x + ck * delta, lb, ub)
        x_minus = np.clip(x - ck * delta, lb, ub)

        y_plus = evaluate_x(x_plus)
        y_minus = evaluate_x(x_minus)

        ghat = ((y_plus - y_minus) / (2.0 * ck)) * delta
        x = np.clip(x + ak * ghat, lb, ub)

        if y_plus > best_score:
            best_score = y_plus
            best_x = x_plus.copy()
        if y_minus > best_score:
            best_score = y_minus
            best_x = x_minus.copy()

        history.append({
            "iter": int(k),
            "eval": int(eval_count["n"]),
            "y_plus": float(y_plus),
            "y_minus": float(y_minus),
            "best_score": float(best_score),
            "ak": float(ak),
            "ck": float(ck),
        })

        if verbose >= 2 and (k == 1 or k % 10 == 0 or k == spsa_steps):
            print(f"[SPSA] iter {k}/{spsa_steps} | best={best_score:.6f}")

    if leftover_eval == 1:
        y_cur = evaluate_x(x)
        history.append({"eval": int(eval_count["n"]), "type": "leftover", "score": float(y_cur)})
        if y_cur > best_score:
            best_score = y_cur
            best_x = x.copy()

    if not np.isfinite(best_score):
        y0 = evaluate_x(x)
        best_score = y0
        best_x = x.copy()
        history.append({"eval": int(eval_count["n"]), "type": "fallback", "score": float(y0)})

    best_counts = x_to_counts(best_x)
    best_actions = counts_to_binary_action_plan(
        counts_block=best_counts,
        num_od=num_od,
        total_step=total_step,
        block_steps=block_steps
    )

    spsa_log = {
        "method": "SPSA",
        "alpha": float(alpha),
        "gamma": float(gamma),
        "a": float(a),
        "c": float(c),
        "A": float(A),
        "init_points": int(init_points),
        "n_iter_eval_budget": int(n_iter),
        "spsa_steps": int(spsa_steps),
        "total_evals": int(eval_count["n"]),
        "history": history,
    }

    return best_actions, float(best_score), spsa_log, best_counts

os.makedirs(Config.RESULT_DIR, exist_ok=True)

_trial_result_dir = RESULT_DIR if "RESULT_DIR" in globals() else Config.RESULT_DIR
reset_trial_times(_trial_result_dir)

for trial in range(5):
    _trial_t0 = time.perf_counter()
    trial_idx = trial
    seed_opt = int(100 + (trial_idx + 1))
                                                                          
    seed_eval = trial

    best_actions, best_score, spsa_log, best_counts = spsa_optimize_counts_300s(
        build_env_fn=build_env,
        worker_idx=0,
        num_od=Config.NUM_OD,
        total_step=Config.TOTAL_STEP,
        input_interval=Config.INPUT_INTERVAL,
        detector_interval=Config.DETECTOR_INTERVAL,
        init_points=0,
        n_iter=600,
        seed=seed_opt,
        verbose=2,

        alpha=0.602,
        gamma=0.101,
        a=3.0,
        c=2.0,
        A=None,
    )

    print(f"[trial {trial}] Best total_reward:", best_score)
    print(f"[trial {trial}] Best action shape:", best_actions.shape)
    print(f"[trial {trial}] Best counts shape:", best_counts.shape)

    replay_reward, trajectory = run_episode_with_plan(
        build_env_fn=build_env,
        worker_idx=0,
        action_plan=best_actions,
        seed=seed_eval
    )
    print(f"[trial {trial}] Replay reward:", replay_reward)

    with open(os.path.join(Config.RESULT_DIR, f"trajectory_{trial}.json"), "w", encoding="utf-8") as f:
        json.dump(
            trajectory, f, ensure_ascii=False, indent=2,
            default=lambda o: o.tolist() if isinstance(o, np.ndarray) else o
        )

    with open(os.path.join(Config.RESULT_DIR, f"spsa_log_{trial}.json"), "w", encoding="utf-8") as f:
        json.dump(spsa_log, f, ensure_ascii=False, indent=2)
    append_trial_time(_trial_result_dir, trial, time.perf_counter() - _trial_t0)


ans: [11.  0.  3.  4.  5.  1. 20. 21.  4.], det: [36.  0. 28. 13.  9.  0. 32. 25.  2.], reward: -168.0
ans: [ 8. 11.  3. 14. 17.  9. 19. 30.  3.], det: [35. 36. 23. 29. 34. 29. 36. 70. 12.], reward: -515.3333129882812
ans: [23.  4. 13. 20. 17. 11. 15. 23.  5.], det: [29. 18. 24. 41. 43. 25. 25. 59. 14.], reward: -349.22222900390625
ans: [15. 10. 10. 13. 17.  5. 11. 26.  2.], det: [20. 15. 29. 40. 36. 17. 30. 48. 22.], reward: -321.1111145019531
ans: [19.  4.  8. 21. 19. 14. 14. 33.  7.], det: [34. 33.  3. 25. 35. 19. 43. 77.  3.], reward: -464.5555419921875
ans: [14. 15. 15. 14. 12.  7. 13. 23.  6.], det: [39. 22. 29. 42. 37. 18. 41. 61. 14.], reward: -521.3333129882812
Environment closed.
ans: [11.  0.  3.  4.  5.  1. 20. 21.  4.], det: [37.  0. 27. 12.  8.  0. 33. 27.  4.], reward: -170.11111450195312
ans: [ 8. 11.  3. 14. 17.  9. 19. 30.  3.], det: [30. 27. 14. 28. 33. 32. 44. 61.  9.], reward: -384.8888854980469
ans: [23.  4. 13. 20. 17. 11. 15. 23.  5.], det: [28. 17. 13. 45. 40. 

In [3]:
import os
import shutil
from config import Config


def remove_workers_root():
                                                            
    root = os.path.abspath(Config.WORKERS_ROOT)
    project_root = os.path.abspath(Config.RL_ROOT)
    if os.path.basename(root) != "workers":
        raise RuntimeError(f"Refusing to delete non-workers path: {root}")
    if os.path.commonpath([root, project_root]) != project_root:
        raise RuntimeError(f"Refusing to delete path outside experiment root: {root}")
    if os.path.isdir(root):
        shutil.rmtree(root)


remove_workers_root()
